### Scott 10K sigdiff (Local Machine version)

#### Only plotting swarmplots
6. Plot swarmplots
    This is the hardest bit because we need swarmplots where all datapoints are plotted
    Need to iterate through various settings
    1 Plot with all datapoints from AD and CN
    1 Plot with those beyond 2 standard deviations cleaned out

#### AD and CN swarmplot

In [23]:
from pathlib import Path

Path("C:/Users/User/Downloads/ephemeral/scott_10k_scripts/sig_diff").mkdir(
    parents=True,
    exist_ok=True
)

Path("C:/Users/User/Downloads/ephemeral/sankeith/scott_10k_logs/sig_diff").mkdir(
    parents=True,
    exist_ok=True
)

In [24]:
%%file C:/Users/User/Downloads/ephemeral/scott_10k_scripts/sig_diff/swarmplots.py

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
from scipy.stats import kstest, levene, mannwhitneyu, ttest_ind
from matplotlib.colors import ListedColormap
from statsmodels.stats.multitest import multipletests
from matplotlib import font_manager

font_path = "C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_housekeeping/arial.ttf"
font_manager.fontManager.addfont(font_path)

swarmplot_path = 'C:/Users/User/Downloads/scott_10k_analysis/swarmplots'
os.makedirs(swarmplot_path, exist_ok = True)

def get_significance_stars(p_value):
    if p_value < 0.001: return '***'
    elif 0.001 < p_value < 0.01: return '**'
    elif 0.01 < p_value < 0.05: return '*'
    else: return 'ns'


for coeff_path in ['C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_housekeeping/desikan_coeffs.csv',
                  'C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_housekeeping/flipped_desikan_coeffs.csv',
                  'C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_housekeeping/destrieux_coeffs.csv',
                  'C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_housekeeping/flipped_destrieux_coeffs.csv']:

    df = pd.read_csv(coeff_path, low_memory=False)
    diag_df = pd.read_csv('C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_housekeeping/no_neuroimaging_data_scott10k_alliedhealth.csv', 
                          low_memory=False)[['DATA_KEY', 'DIAGNOSIS']]

    merged = pd.merge(diag_df, df, on='DATA_KEY', how='inner')
    cn_ad_df = merged[merged['DIAGNOSIS'].isin([1, 3])].drop(columns = 'DATA_KEY')

    melted_df = cn_ad_df.melt(id_vars='DIAGNOSIS', var_name='NEUROTRANSMITTER', value_name='VALUE')

    print(melted_df)

    stats_df = pd.read_csv(
        f"C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_analysis/sig_diff/{os.path.basename(coeff_path).replace('s.csv', '')}_sig_diff.csv",
        low_memory = False)


    for width in range(50, 60):
        for dot_size in np.arange(0.1, 2.1, 0.1):
            height = (width / 4) 

            print(f"PRINTING SWARMPLOT - WIDTH = {width}, HEIGHT = {height}, DOT SIZE = {dot_size}")

            pval_dict = dict(zip(stats_df['Coefficient'], stats_df['Corrected_P']))

            print(pval_dict)

            assert list(pval_dict.keys()) == cn_ad_df.drop(columns = 'DIAGNOSIS').columns.tolist()

            
            plt.figure(figsize=(width, height))
            
            ax = sns.swarmplot(
                data=melted_df, 
                x = 'NEUROTRANSMITTER', 
                y = 'VALUE', 
                hue ='DIAGNOSIS', 
                palette = {1: 'green', 3: 'red'},
                dodge = True, 
                size = dot_size, 
                alpha = 0.7
            )

            plt.axhline(y = 0, color = 'grey', linestyle = 'dashed')

            # Order + y positions
            x_order = melted_df['NEUROTRANSMITTER'].unique()

            print(x_order)
            y_max = melted_df.groupby('NEUROTRANSMITTER')['VALUE'].max()
            print(y_max)

            y_range = melted_df['VALUE'].max() - melted_df['VALUE'].min()
            bracket_height = 0.01 * y_range  # How tall the "tips" of the bracket are
            bracket_offset = 0.02 * y_range  # Space between data and bracket
            text_offset = 0.005 * y_range 
            
            # Add stars
            for i, nt in enumerate(x_order):
                if nt in pval_dict:
                    stars = get_significance_stars(pval_dict[nt])
                    
                    if stars != 'ns':
                        y_base = y_max[nt] + bracket_offset
                        x1, x2 = i - 0.2, i + 0.2

                        ax.plot([x1, x1, x2, x2], 
                                [y_base - bracket_height, y_base, y_base, y_base - bracket_height], 
                                lw=1.5, color='black')

                        ax.text((x1 + x2) * .5, y_base + text_offset, stars, 
                                ha='center', va='bottom', fontsize=12, fontweight='bold')
                        
                        y = y_max[nt] + 0.02 * (y_max.max() - y_max.min())
                        ax.text(i, y, stars, ha='center', va='bottom', fontsize=10)

            plt.tight_layout()
            final_path = os.path.join(swarmplot_path, f"{os.path.basename(coeff_path).replace('_coeffs.csv', '')}_w{width}_h{height}_d{dot_size:.1f}.png")
            print(final_path)
            plt.savefig(final_path, dpi = 300)
            plt.close()

 

Overwriting C:/Users/User/Downloads/ephemeral/scott_10k_scripts/sig_diff/swarmplots.py


In [32]:
%%file C:/Users/User/Downloads/ephemeral/scott_10k_scripts/sig_diff/para_swarmplots.py

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys
from matplotlib import font_manager

font_path = "C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_housekeeping/arial.ttf"
font_manager.fontManager.addfont(font_path)

swarmplot_path = 'C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_analysis/swarmplots'
os.makedirs(swarmplot_path, exist_ok=True)

def get_significance_stars(p_value):
    if p_value < 0.001: return '***'
    elif p_value < 0.01: return '**'
    elif p_value < 0.05: return '*'
    else: return 'ns'

coeff_options = [
    'desikan_coeffs.csv', 'flipped_desikan_coeffs.csv',
    'destrieux_coeffs.csv', 'flipped_destrieux_coeffs.csv'
]
selected_filename = coeff_options[int(sys.argv[1])]
coeff_path = os.path.join('C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_housekeeping/', selected_filename)

df = pd.read_csv(coeff_path, low_memory=False)
diag_df = pd.read_csv('C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_housekeeping/no_neuroimaging_data_scott10k_alliedhealth.csv', 
                      low_memory=False)[['DATA_KEY', 'DIAGNOSIS']]

merged = pd.merge(diag_df, df, on='DATA_KEY', how='inner')
cn_ad_df = merged[merged['DIAGNOSIS'].isin([1, 3])].drop(columns='DATA_KEY')
melted_df = cn_ad_df.melt(id_vars='DIAGNOSIS', var_name='NEUROTRANSMITTER', value_name='VALUE')

# 3. Outlier Filtering (Ensures the plot isn't squashed by extreme values)
group_stats = melted_df.groupby(['NEUROTRANSMITTER', 'DIAGNOSIS'])['VALUE']
means = group_stats.transform('mean')
stds = group_stats.transform('std')

# 4. Global Stats (Calculate ONCE before loops)
v_min = melted_df['VALUE'].min()
v_max = melted_df['VALUE'].max()
y_range = v_max - v_min
x_order = melted_df['NEUROTRANSMITTER'].unique()
y_max_per_nt = melted_df.groupby('NEUROTRANSMITTER')['VALUE'].max()

print(f"C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_analysis/sig_diff/{selected_filename.replace('.csv', '')}_sig_diff.csv")
stats_df = pd.read_csv(
    f"C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_analysis/sig_diff/{selected_filename.replace('.csv', '')}_sig_diff.csv",
    low_memory=False)
pval_dict = dict(zip(stats_df['Coefficient'], stats_df['Corrected_P']))

# 5. Plotting Loops
for width in range(61, 65, 1):
    for dot_size in [0.6, 0.7, 1]:
        height = max(6, float(width / 4))
        print(f"Plotting {selected_filename}: W={width}, H={height:.1f}, Size={dot_size}")

        plt.figure(figsize=(width, height))

        ax = sns.swarmplot(
            data = melted_df, 
            x='NEUROTRANSMITTER', 
            y = 'VALUE',
            hue = 'DIAGNOSIS', 
            palette={1: 'green', 3: 'red'},
            dodge=True, 
            size=dot_size, 
            alpha=0.7,
            order=x_order
        )

        ax.set_yscale("linear")
        plt.axhline(y = 0, color = 'grey', linestyle = 'dashed')
        plt.xlabel('Neurotransmitter map')
        plt.ylabel('Beta coefficient value')
        # Add 5% padding at top for stars
        y_range = v_max - v_min
        padding = y_range * 0.05
        ax.set_ylim(v_min - padding, v_max + padding)

        # 6. Add Brackets and Stars

        y_total_range = v_max - v_min # Fix 1: Use existing global min/max

        for i, nt in enumerate(x_order):
            if nt in pval_dict:
                stars = get_significance_stars(pval_dict[nt])

                if stars != 'ns':
                    # Fix 2: Use y_max_per_nt variable defined in step 4
                    y_base = y_max_per_nt[nt] + 0.02 * y_total_range  
                    y_tips = y_base - 0.01 * y_total_range    

                    x1, x2 = i - 0.2, i + 0.2

                    ax.plot([x1, x1, x2, x2], [y_tips, y_base, y_base, y_tips], lw=1, color='black')
                    ax.text((x1 + x2) / 2, y_base, stars, ha='center', va='bottom', fontsize=10)

        plt.title(f"Group Comparison: {selected_filename.replace('_coeffs.csv', '')} (Linear Scale)")
        sns.despine()
        plt.legend(['CN', 'AD'], loc = 'upper right', title='Diagnosis')  
        plt.tight_layout()
        save_name = f"{selected_filename.replace('_coeffs.csv', '')}_linear_w{width}_h{height:.1f}_d{dot_size:.2f}.png"
        plt.savefig(os.path.join(swarmplot_path, save_name), dpi=300)
        plt.close()


Overwriting C:/Users/User/Downloads/ephemeral/scott_10k_scripts/sig_diff/para_swarmplots.py


#### Swarmplots - but data is cleaned to be within 2 standard deviations for each neurotransmitter's distribution

In [33]:
%%file C:/Users/User/Downloads/ephemeral/scott_10k_scripts/sig_diff/para_c_swarmplots.py

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys
from matplotlib import font_manager

font_path = "C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_housekeeping/arial.ttf"
font_manager.fontManager.addfont(font_path)


swarmplot_path = 'C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_analysis/swarmplots'
os.makedirs(swarmplot_path, exist_ok=True)

def get_significance_stars(p_value):
    if p_value < 0.001: return '***'
    elif p_value < 0.01: return '**'
    elif p_value < 0.05: return '*'
    else: return 'ns'

coeff_options = [
    'desikan_coeffs.csv', 'flipped_desikan_coeffs.csv',
    'destrieux_coeffs.csv', 'flipped_destrieux_coeffs.csv'
]
selected_filename = coeff_options[int(sys.argv[1])]
coeff_path = os.path.join('C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_housekeeping/', selected_filename)

df = pd.read_csv(coeff_path, low_memory=False)
diag_df = pd.read_csv('C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_housekeeping/no_neuroimaging_data_scott10k_alliedhealth.csv', 
                      low_memory=False)[['DATA_KEY', 'DIAGNOSIS']]

merged = pd.merge(diag_df, df, on='DATA_KEY', how='inner')
cn_ad_df = merged[merged['DIAGNOSIS'].isin([1, 3])].drop(columns='DATA_KEY')
melted_df = cn_ad_df.melt(id_vars='DIAGNOSIS', var_name='NEUROTRANSMITTER', value_name='VALUE')

group_stats = melted_df.groupby(['NEUROTRANSMITTER', 'DIAGNOSIS'])['VALUE']
means = group_stats.transform('mean')
stds = group_stats.transform('std')
filtered_df = melted_df[(melted_df['VALUE'] >= means - 2*stds) & (melted_df['VALUE'] <= means + 2*stds)].copy()

v_min = filtered_df['VALUE'].min()
v_max = filtered_df['VALUE'].max()
y_range = v_max - v_min
x_order = filtered_df['NEUROTRANSMITTER'].unique()
y_max_per_nt = filtered_df.groupby('NEUROTRANSMITTER')['VALUE'].max()

print(f"C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_analysis/sig_diff/{selected_filename.replace('.csv', '')}_sig_diff.csv")
stats_df = pd.read_csv(
    f"C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_analysis/sig_diff/{selected_filename.replace('.csv', '')}_sig_diff.csv",
    low_memory=False)
pval_dict = dict(zip(stats_df['Coefficient'], stats_df['Corrected_P']))

for width in range(61, 65, 1):
    for dot_size in [0.6, 0.7, 1]:
        height = max(6, float(width / 4))
        print(f"Plotting {selected_filename}: W={width}, H={height:.1f}, Size={dot_size}")

        plt.figure(figsize=(width, height))
        
        ax = sns.swarmplot(
            data=filtered_df, 
            x='NEUROTRANSMITTER', 
            y= 'VALUE',
            hue='DIAGNOSIS', 
            palette={1: 'green', 3: 'red'},
            dodge=True, 
            size=dot_size, 
            alpha=0.7,
            order=x_order
        )

        ax.set_yscale("linear")
        plt.axhline(y = 0, color = 'grey', linestyle = 'dashed')
        plt.xlabel('Neurotransmitter map')
        plt.ylabel('Beta coefficient value')
        # Add 5% padding at top for stars
        y_range = v_max - v_min
        padding = y_range * 0.05
        ax.set_ylim(v_min - padding, v_max + padding)

        # 6. Add Brackets and Stars
        
        y_total_range = v_max - v_min # Fix 1: Use existing global min/max
            
        for i, nt in enumerate(x_order):
            if nt in pval_dict:
                stars = get_significance_stars(pval_dict[nt])
                
                if stars != 'ns':
                    # Fix 2: Use y_max_per_nt variable defined in step 4
                    y_base = y_max_per_nt[nt] + 0.02 * y_total_range  
                    y_tips = y_base - 0.01 * y_total_range    
                    
                    x1, x2 = i - 0.2, i + 0.2
                    
                    ax.plot([x1, x1, x2, x2], [y_tips, y_base, y_base, y_tips], lw=1, color='black')
                    ax.text((x1 + x2) / 2, y_base, stars, ha='center', va='bottom', fontsize=10)

        plt.title(f"Group Comparison: {selected_filename.replace('_coeffs.csv', '')} (Linear Scale)")
        sns.despine()
        plt.legend(['CN', 'AD'], loc = 'upper right', title='Diagnosis')  
        plt.tight_layout()
        save_name = f"{selected_filename.replace('_coeffs.csv', '')}_c_linear_w{width}_h{height:.1f}_d{dot_size:.2f}.png"
        plt.savefig(os.path.join(swarmplot_path, save_name), dpi=300)
        plt.close()

Writing C:/Users/User/Downloads/ephemeral/scott_10k_scripts/sig_diff/para_c_swarmplots.py


#### Some code for specifically plotting desikan_c with different dot sizes


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys
from matplotlib import font_manager

font_path = "C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_housekeeping/arial.ttf"
font_manager.fontManager.addfont(font_path)


swarmplot_path = 'C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_analysis/swarmplots'
os.makedirs(swarmplot_path, exist_ok=True)

def get_significance_stars(p_value):
    if p_value < 0.001: return '***'
    elif p_value < 0.01: return '**'
    elif p_value < 0.05: return '*'
    else: return 'ns'

selected_filename = 'desikan_coeffs.csv'
coeff_path = os.path.join('C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_housekeeping/', selected_filename)

df = pd.read_csv(coeff_path, low_memory=False)
diag_df = pd.read_csv('C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_housekeeping/no_neuroimaging_data_scott10k_alliedhealth.csv', 
                      low_memory=False)[['DATA_KEY', 'DIAGNOSIS']]

merged = pd.merge(diag_df, df, on='DATA_KEY', how='inner')
cn_ad_df = merged[merged['DIAGNOSIS'].isin([1, 3])].drop(columns='DATA_KEY')
melted_df = cn_ad_df.melt(id_vars='DIAGNOSIS', var_name='NEUROTRANSMITTER', value_name='VALUE')

group_stats = melted_df.groupby(['NEUROTRANSMITTER', 'DIAGNOSIS'])['VALUE']
means = group_stats.transform('mean')
stds = group_stats.transform('std')
filtered_df = melted_df[(melted_df['VALUE'] >= means - 2*stds) & (melted_df['VALUE'] <= means + 2*stds)].copy()

v_min = filtered_df['VALUE'].min()
v_max = filtered_df['VALUE'].max()
y_range = v_max - v_min
x_order = filtered_df['NEUROTRANSMITTER'].unique()
y_max_per_nt = filtered_df.groupby('NEUROTRANSMITTER')['VALUE'].max()

print(f"C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_analysis/sig_diff/{selected_filename.replace('.csv', '')}_sig_diff.csv")
stats_df = pd.read_csv(
    f"C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_analysis/sig_diff/{selected_filename.replace('.csv', '')}_sig_diff.csv",
    low_memory=False)
pval_dict = dict(zip(stats_df['Coefficient'], stats_df['Corrected_P']))

width = 61
height = max(6, float(width / 4))
for dot_size in [1.4]:
        
    print(f"Plotting {selected_filename}: W={width}, H={height:.1f}, Size={dot_size}")

    plt.figure(figsize=(width, height))
    
    ax = sns.swarmplot(
        data=filtered_df, 
        x='NEUROTRANSMITTER', 
        y= 'VALUE',
        hue='DIAGNOSIS', 
        palette={1: 'green', 3: 'red'},
        dodge=True, 
        size=dot_size, 
        alpha=0.7,
        order=x_order
    )

    ax.set_yscale("linear")
    plt.axhline(y = 0, color = 'grey', linestyle = 'dashed')
    plt.xlabel('Neurotransmitter map')
    plt.ylabel('Beta coefficient value')
    # Add 5% padding at top for stars
    y_range = v_max - v_min
    padding = y_range * 0.05
    ax.set_ylim(v_min - padding, v_max + padding)

    # 6. Add Brackets and Stars
    
    y_total_range = v_max - v_min # Fix 1: Use existing global min/max
        
    for i, nt in enumerate(x_order):
        if nt in pval_dict:
            stars = get_significance_stars(pval_dict[nt])
            
            if stars != 'ns':
                # Fix 2: Use y_max_per_nt variable defined in step 4
                y_base = y_max_per_nt[nt] + 0.02 * y_total_range  
                y_tips = y_base - 0.01 * y_total_range    
                
                x1, x2 = i - 0.2, i + 0.2
                
                ax.plot([x1, x1, x2, x2], [y_tips, y_base, y_base, y_tips], lw=1, color='black')
                ax.text((x1 + x2) / 2, y_base, stars, ha='center', va='bottom', fontsize=12)

    plt.title(f"Group Comparison: {selected_filename.replace('_coeffs.csv', '')} (Linear Scale)")
    sns.despine()
    plt.legend(['CN', 'AD'], loc = 'upper right', title='Diagnosis')  
    plt.tight_layout()
    save_name = f"{selected_filename.replace('_coeffs.csv', '')}_c_linear_w{width}_h{height:.1f}_d{dot_size:.2f}.png"
    plt.savefig(os.path.join(swarmplot_path, save_name), dpi=300)
    plt.close()

C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_analysis/sig_diff/desikan_coeffs_sig_diff.csv
Plotting desikan_coeffs.csv: W=61, H=15.2, Size=1.4
